# Sliding-window intertextuality

For each experiment, a window slides across the corpus and each model rates how closely every window relates to the query, from 0 to 10. Choose *Runtime → Change runtime type → GPU*, then *Runtime → Run all*; the results download as a zip at the end.

In [ ]:
!pip install -q -U "transformers>=5.23" accelerate bitsandbytes openpyxl

## Settings

In [ ]:
import json

# The texts, experiments and models chosen on the website.
DATA = json.loads("__DATA__")

MODELS = DATA["models"]            # Hugging Face model ids, run in this order
WINDOW_WORDS = 40                  # words per window (40 Greek words cover about as much text as 50 English words)
STRIDE_WORDS = 4                   # how far the window moves each step
INCLUDE_LEXICAL_BASELINE = True    # a word-overlap reference line on the graphs, computed without any model
FORCE_4BIT = None                  # None: 4-bit only when a model would not fit in 16-bit. True/False to force.
MIN_DIGIT_MASS = 0.5               # warn when a model puts less probability than this on answering with a number
OUTPUT_DIR = "intertextuality_results"
DRY_RUN = False                    # True: fake scores and no model download, to test the notebook quickly

In [ ]:
import os, re, gc, time, math, random, zlib, pathlib, unicodedata, shutil
from collections import Counter
import numpy as np
import pandas as pd

try:
    import torch
    import transformers
except ImportError:
    torch = transformers = None
    if not DRY_RUN:
        raise

if transformers is not None and not DRY_RUN:
    if tuple(int(x) for x in re.findall(r"\d+", transformers.__version__)[:2]) < (5, 23):
        raise RuntimeError(f"transformers {transformers.__version__} is loaded, but 5.23 or newer is needed. "
                           "Runtime → Restart session, then Run all again.")

DEVICE = "cuda" if (torch is not None and torch.cuda.is_available()) else "cpu"
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if DEVICE == "cuda" else 0.0
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)} ({VRAM_GB:.0f} GB)")
elif not DRY_RUN:
    print("WARNING: no GPU. In Colab: Runtime → Change runtime type → GPU.")

def colab_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

try:
    _token = colab_secret("HF_TOKEN")     # optional: a Colab secret named HF_TOKEN, needed only for gated models
except Exception:
    _token = None
if _token:
    os.environ["HF_TOKEN"] = _token

OUT = pathlib.Path(OUTPUT_DIR)
OUT.mkdir(parents=True, exist_ok=True)
RESULTS_PATH = OUT / "window_scores.jsonl"
pd.set_option("display.max_colwidth", 90)

## Prompt

In [ ]:
PROMPT_TEMPLATE = """Compare the two passages below.

QUERY:
{query}

EXCERPT:
{excerpt}

The EXCERPT is a short run of words cut from a longer text, so it may begin or end in the middle of a sentence. How closely related are the EXCERPT and the QUERY? Look for quotation or close paraphrase, shared distinctive words and phrases (especially rare ones), and distinctive shared images or ideas. Common words and generic religious language count for little.

0 = unrelated
1-3 = only superficial or generic overlap
4-6 = a probable allusion or echo: distinctive shared words, phrases or ideas
7-8 = a strong allusion with several distinctive shared words or phrases
9-10 = one passage quotes or closely paraphrases the other

Answer with a single integer from 0 to 10 and nothing else."""

## Experiments and windows

The model sees only the text of each passage, never the references.

In [ ]:
def nfc(s):
    return unicodedata.normalize("NFC", str(s))

def passage(t):
    rows = [(nfc(r.get("reference") or "").strip() or f"row {i}", " ".join(nfc(r.get("text", "")).split()))
            for i, r in enumerate(t["rows"], 1)]
    rows = [(ref, text) for ref, text in rows if text]
    return {"name": t["name"], "refs": [r for r, _ in rows], "texts": [x for _, x in rows]}

def common_prefix(refs):
    """Shared beginning of the references, cut back to a space, colon or full stop ("Psalm 25:" for Psalm 25:1-22)."""
    if len(refs) < 2:
        return ""
    p = os.path.commonprefix(list(refs))
    cut = max(p.rfind(":"), p.rfind(" "), p.rfind("."))
    return p[:cut + 1] if cut >= 0 and all(len(r) > cut + 1 for r in refs) else ""

def make_windows(n, size, stride):
    size = max(1, min(size, n))
    starts = list(range(0, n - size + 1, max(1, stride)))
    if starts[-1] + size < n:
        starts.append(n - size)
    return [(s, s + size) for s in starts]

EXPERIMENTS, titles = [], set()
for i, e in enumerate(DATA["experiments"], 1):
    q, c = passage(DATA["texts"][e["query"]]), passage(DATA["texts"][e["corpus"]])
    title, k = f"{q['name']} | {c['name']}", 2
    while title in titles:
        title, k = f"{q['name']} | {c['name']} ({k})", k + 1
    titles.add(title)
    words, verse_idx = [], []
    for v, text in enumerate(c["texts"]):
        for w in text.split():
            words.append(w)
            verse_idx.append(v)
    prefix = common_prefix(c["refs"])
    short = [r[len(prefix):] for r in c["refs"]]
    windows = []
    for n, (s, t) in enumerate(make_windows(len(words), WINDOW_WORDS, STRIDE_WORDS), 1):
        a, b = c["refs"][verse_idx[s]], c["refs"][verse_idx[t - 1]]
        windows.append({"window": n, "start": s, "end": t, "verses": a if a == b else f"{a} – {short[verse_idx[t - 1]]}",
                        "text": " ".join(words[s:t])})
    query_text = " ".join(q["texts"])
    fp = f"exp{i:02d}-" + format(zlib.crc32(json.dumps([PROMPT_TEMPLATE, query_text, words, WINDOW_WORDS, STRIDE_WORDS],
                                                       ensure_ascii=False).encode()), "08x")
    EXPERIMENTS.append({"id": f"exp{i:02d}", "title": title, "query": q, "corpus": c, "query_text": query_text,
                        "words": words, "verse_idx": np.array(verse_idx), "short_refs": short, "prefix": prefix,
                        "windows": windows, "fp": fp})
EXP = {e["id"]: e for e in EXPERIMENTS}

def build_prompt(e, w):
    return PROMPT_TEMPLATE.format(query=e["query_text"], excerpt=w["text"])

display(pd.DataFrame([{"id": e["id"], "experiment": e["title"], "query words": len(e["query_text"].split()),
                       "corpus words": len(e["words"]), "windows": len(e["windows"])} for e in EXPERIMENTS]).set_index("id"))
print("Models:", ", ".join(MODELS) or "none")

## Scorer

Each window gets one forward pass. The score is the probability-weighted average of the answers 0–10, read from the probability the model gives each possible first token. A model that writes "10" as "1" then "0" gets a "1" appended to the same pass, to read how likely "0" comes next. Thinking is switched off (`enable_thinking=False`), so the answer starts straight away.

In [ ]:
def model_params_b(hf_id):
    """Parameter count in billions from the Hugging Face Hub, or None if it cannot be found."""
    try:
        from huggingface_hub import HfApi
        info = HfApi().model_info(hf_id, files_metadata=True)
        st = getattr(info, "safetensors", None)
        if st is not None and getattr(st, "total", None):
            return st.total / 1e9
        size = sum((s.size or 0) for s in info.siblings if s.rfilename.endswith((".safetensors", ".bin")))
        return size / 2e9 if size else None
    except Exception:
        return None

def plan_loading(hf_id):
    p = model_params_b(hf_id)
    cfg = transformers.AutoConfig.from_pretrained(hf_id)
    prequantized = any(getattr(x, "quantization_config", None) for x in (cfg, getattr(cfg, "text_config", None)) if x)
    if DEVICE != "cuda":
        return {"quantize": False, "dtype": torch.float32, "note": "CPU, float32", "params": p, "gb": None,
                "fits": p is None or p <= 8}
    budget = 0.8 * VRAM_GB
    if torch.cuda.get_device_capability()[0] >= 8:         # Ampere or newer: native bfloat16
        dtype, name = torch.bfloat16, "bfloat16"
    else:
        dtype, name = torch.float16, "float16"
    if prequantized:
        return {"quantize": False, "dtype": dtype, "note": f"the checkpoint's own quantisation, {name}", "params": p,
                "gb": None, "fits": True}
    too_big = p is not None and p * 2 + 2.5 > budget
    quantize = too_big if FORCE_4BIT is None else bool(FORCE_4BIT)
    gb = None if p is None else (p * 0.6 + 2.0 if quantize else p * 2 + 1.5)
    return {"quantize": quantize, "dtype": dtype, "note": f"4-bit NF4, {name} compute" if quantize else name,
            "params": p, "gb": gb, "fits": gb is None or gb < budget}


class LocalScorer:
    def __init__(self, hf_id):
        self.hf_id = hf_id
        plan = plan_loading(hf_id)
        size = f"{plan['params']:.1f}B parameters, " if plan["params"] else ""
        print(f"  {size}loading as {plan['note']}" + (f", about {plan['gb']:.0f} GB" if plan["gb"] else ""))
        if not plan["fits"]:
            raise RuntimeError(f"needs about {plan['gb']:.0f} GB of GPU memory, but this GPU has {VRAM_GB:.0f} GB. "
                               "Choose a larger GPU (Runtime → Change runtime type).")
        kw = {"dtype": plan["dtype"], "output_loading_info": True}
        if DEVICE == "cuda":
            kw["device_map"] = "auto"
        if plan["quantize"]:
            kw["quantization_config"] = transformers.BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=plan["dtype"],
                bnb_4bit_use_double_quant=True)
        t0 = time.time()
        self.tok = transformers.AutoTokenizer.from_pretrained(hf_id)
        model, info, errors = None, None, []
        # Qwen3.5 and later are image-text-to-text models (only the text part runs); others are causal LMs.
        for cls_name in ("AutoModelForImageTextToText", "AutoModelForCausalLM"):
            try:
                model, info = getattr(transformers, cls_name).from_pretrained(hf_id, **kw)
                break
            except (ValueError, KeyError) as e:
                errors.append(f"{cls_name}: {e}")
        if model is None:
            raise RuntimeError("could not load the model:\n" + "\n".join(errors))
        missing = sorted(info.get("missing_keys") or [])
        if missing:
            raise RuntimeError(f"{len(missing)} weights are missing from the checkpoint (e.g. {missing[:3]}).")
        self.model = model.eval()
        self.device = next(self.model.parameters()).device
        if any(v in ("cpu", "disk") for v in (getattr(model, "hf_device_map", None) or {}).values()):
            print("  WARNING: part of the model did not fit on the GPU and runs on the CPU; this will be very slow.")
        print(f"  loaded in {time.time() - t0:.0f}s")
        self._find_answer_tokens()
        tail = self.prompt_text("test")
        if "<think>" in str(getattr(self.tok, "chat_template", "")) and not tail.rstrip().endswith("</think>"):
            raise RuntimeError("thinking could not be switched off in this model's chat template.")

    def _single(self, text):
        ids = self.tok.encode(text, add_special_tokens=False)
        return ids[0] if len(ids) == 1 and self.tok.decode(ids).strip() == text.strip() else None

    def _find_answer_tokens(self):
        self.answer_ids = {v: sorted({i for i in (self._single(str(v)), self._single(" " + str(v))) if i is not None})
                           for v in range(11)}
        self.one_id, self.zero_id = self._single("1"), self._single("0")
        self.split_ten = not self.answer_ids[10]
        if any(not self.answer_ids[v] for v in range(10)) or self.one_id is None or self.zero_id is None:
            raise RuntimeError("this tokenizer does not write the digits 0-9 as single tokens.")

    def prompt_text(self, prompt):
        if not getattr(self.tok, "chat_template", None):          # a base model without a chat template
            return prompt + "\n\nAnswer:"
        return self.tok.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False,
                                            add_generation_prompt=True, enable_thinking=False)

    def prompt_ids(self, prompt):
        return self.tok(self.prompt_text(prompt), add_special_tokens=False)["input_ids"]

    def score(self, prompt):
        ids = self.prompt_ids(prompt) + [self.one_id]
        x = torch.tensor([ids], device=self.device)
        with torch.inference_mode():
            out = self.model(input_ids=x, attention_mask=torch.ones_like(x), logits_to_keep=2, use_cache=False)
        logits = out.logits[0, -2:, :].float()
        p_first = torch.softmax(logits[0], -1)
        mass = {v: float(p_first[self.answer_ids[v]].sum()) if self.answer_ids[v] else 0.0 for v in range(11)}
        if self.split_ten:
            p_zero = float(torch.softmax(logits[1], -1)[self.zero_id])
            mass[10] += mass[1] * p_zero
            mass[1] *= 1 - p_zero
        total = sum(mass.values())
        probs = [mass[v] / total for v in range(11)] if total > 0 else [float("nan")] * 11
        score = sum(v * p for v, p in enumerate(probs))
        if math.isnan(score):
            raise RuntimeError("the model returned NaN probabilities (a float16 overflow?). "
                               "Use an L4, A100 or H100 runtime, which computes in bfloat16.")
        return {"score": score, "top": int(max(mass, key=mass.get)), "digit_mass": total, "probs": probs}

    def reply(self, prompt, max_new_tokens=8):
        """What the model actually writes (greedy), as a check that it answers with a number."""
        x = torch.tensor([self.prompt_ids(prompt)], device=self.device)
        with torch.inference_mode():
            out = self.model.generate(input_ids=x, attention_mask=torch.ones_like(x), max_new_tokens=max_new_tokens,
                                      do_sample=False, pad_token_id=self.tok.pad_token_id or self.tok.eos_token_id)
        return self.tok.decode(out[0, x.shape[1]:], skip_special_tokens=True).strip()


STOPWORDS = set("""
ο η το του τησ τω τη τον την οι αι τα των τοισ ταισ τουσ τασ και δε τε γαρ ουν μεν αν εν εισ εκ εξ απο επι
προσ δια κατα μετα περι υπο υπερ παρα συν αλλα αλλ ινα οτι ωσ ουτωσ ουτω μη ου ουκ ουχ ει εαν αυτοσ αυτου
αυτω αυτον αυτοι αυτων αυτοισ αυτουσ αυτη αυτην αυτησ μου μοι με εμε σου σοι σε υμιν υμων υμασ υμεισ ημιν
ημων ημασ ημεισ εγω συ τισ τι οσ ω ον ην α ουσ ων οισ εστιν εισιν εστι ουδε μηδε ουτοσ τουτο ταυτα τουτου
the a an and of to in that it is be ye you your yours thy thine thee thou for with not shall unto which them
they their he his him i my mine me we us our by as so but or are was were there this these those what will
at on from into upon all if then than when who whom also hath have had doth let therefore out many
""".split())

def norm_words(text):
    t = unicodedata.normalize("NFD", text.lower())
    t = "".join(ch for ch in t if not unicodedata.combining(ch)).replace("ς", "σ")
    return [w for w in re.findall(r"[^\W\d_]+", t) if len(w) > 1 and w not in STOPWORDS]

def lexical_scores(e):
    """Share of each window's distinctive (rarer) words that also occur in the query, times 10."""
    query = set(norm_words(e["query_text"]))
    sets = [set(norm_words(w["text"])) for w in e["windows"]]
    df_ = Counter(x for s in sets for x in s)
    idf = {x: math.log((len(sets) + 1) / (df_[x] + 1)) + 1 for x in df_}
    return [10 * sum(idf[x] for x in s & query) / sum(idf[x] for x in s) if s else 0.0 for s in sets]


class DryRunScorer:
    """Fake model for testing: the lexical score, stretched and jittered differently for each model."""
    def __init__(self, name):
        self.rng = random.Random(zlib.crc32(name.encode()))
        self.gain, self.shift = self.rng.uniform(1.2, 2.2), self.rng.uniform(0.5, 3.0)
        self.lex = {build_prompt(e, w): s for e in EXPERIMENTS for w, s in zip(e["windows"], lexical_scores(e))}

    def score(self, prompt):
        s = float(np.clip(self.shift + self.gain * self.lex[prompt] + self.rng.gauss(0, 0.6), 0, 10))
        p = np.exp(-0.5 * ((np.arange(11) - s) / 1.2) ** 2)
        return {"score": s, "top": int(round(s)), "digit_mass": self.rng.uniform(0.6, 0.99), "probs": list(p / p.sum())}

    def reply(self, prompt):
        return str(round(self.score(prompt)["score"]))

## Score every window

Scores are saved as they are computed, so after a disconnect *Run all* carries on where it stopped.

In [ ]:
def done_keys():
    keys = set()
    if RESULTS_PATH.exists():
        with RESULTS_PATH.open(encoding="utf-8") as f:
            for line in f:
                r = json.loads(line)
                keys.add((r["model_id"], r["fp"], r["window"]))
    return keys

MODEL_INFO_PATH = OUT / "models.json"
MODEL_INFO = json.loads(MODEL_INFO_PATH.read_text()) if MODEL_INFO_PATH.exists() else {}

def model_label(hf_id):
    return hf_id.rstrip("/").split("/")[-1]

for hf_id in MODELS:
    print(f"\n=== {hf_id} ===")
    done = done_keys()
    todo_total = sum(1 for e in EXPERIMENTS for w in e["windows"] if (hf_id, e["fp"], w["window"]) not in done)
    if not todo_total:
        print("  everything already scored")
        continue
    try:
        scorer = DryRunScorer(hf_id) if DRY_RUN else LocalScorer(hf_id)
    except Exception as err:
        print(f"  could not load {hf_id}: {type(err).__name__}: {err}")
        continue
    info = MODEL_INFO.setdefault(hf_id, {"first_replies": {}})
    for e in EXPERIMENTS:
        todo = [w for w in e["windows"] if (hf_id, e["fp"], w["window"]) not in done]
        if not todo:
            continue
        if e["id"] not in info["first_replies"]:
            try:
                info["first_replies"][e["id"]] = scorer.reply(build_prompt(e, e["windows"][0]))
            except Exception as err:
                info["first_replies"][e["id"]] = f"(could not generate: {err})"
            MODEL_INFO_PATH.write_text(json.dumps(MODEL_INFO, ensure_ascii=False, indent=1))
        t0, rows, failures = time.time(), [], 0
        for w in todo:
            t1 = time.time()
            try:
                res = scorer.score(build_prompt(e, w))
                failures = 0
            except Exception as err:
                failures += 1
                print(f"  {e['id']} window {w['window']} failed: {type(err).__name__}: {err}")
                if failures >= 3:
                    print("  three failures in a row; skipping the rest of this experiment")
                    break
                continue
            row = {"model_id": hf_id, "model": model_label(hf_id), "fp": e["fp"], "experiment": e["id"],
                   "window": w["window"], "start": w["start"], "end": w["end"], "verses": w["verses"], **res,
                   "seconds": round(time.time() - t1, 3)}
            with RESULTS_PATH.open("a", encoding="utf-8") as f:
                f.write(json.dumps(row, ensure_ascii=False) + "\n")
            rows.append(row)
        if rows:
            best = max(rows, key=lambda r: r["score"])
            print(f"  {e['id']} {e['title']}: {len(rows)} windows in {time.time() - t0:.0f}s | max {best['score']:.1f} "
                  f"at {best['verses']} | first reply “{info['first_replies'][e['id']]}”")
    del scorer
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

## Results

### Window rankings

In [ ]:
LEX = "Lexical overlap (no model)"
rows = pd.read_json(RESULTS_PATH, lines=True) if RESULTS_PATH.exists() and RESULTS_PATH.stat().st_size else pd.DataFrame()
if len(rows):
    rows = rows[rows["fp"].isin([e["fp"] for e in EXPERIMENTS]) & rows["model_id"].isin(MODELS)]
    rows = rows.drop_duplicates(["model_id", "fp", "window"], keep="last").copy()
LABEL = {m: model_label(m) if [model_label(x) for x in MODELS].count(model_label(m)) == 1 else m for m in MODELS}
if len(rows):
    rows["label"] = rows["model_id"].map(LABEL)
MODEL_LABELS = [LABEL[m] for m in MODELS if len(rows) and (rows["model_id"] == m).any()]
if not MODEL_LABELS:
    print("No model scores yet: see the messages above.")

lex_rows = []
if INCLUDE_LEXICAL_BASELINE:
    for e in EXPERIMENTS:
        for w, s in zip(e["windows"], lexical_scores(e)):
            lex_rows.append({"model_id": "", "model": LEX, "label": LEX, "fp": e["fp"], "experiment": e["id"],
                             "window": w["window"], "start": w["start"], "end": w["end"], "verses": w["verses"],
                             "score": s})
scores = pd.concat([rows, pd.DataFrame(lex_rows)], ignore_index=True) if len(rows) else pd.DataFrame(lex_rows)
LABELS = MODEL_LABELS + ([LEX] if INCLUDE_LEXICAL_BASELINE else [])
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#6250d6", "#d6336c", "#b88400"]
COLOR = {l: PALETTE[i % len(PALETTE)] for i, l in enumerate(MODEL_LABELS)}
COLOR[LEX] = "#898781"

rank_rows = []
for e in EXPERIMENTS:
    for l in MODEL_LABELS:
        sub = scores[(scores["experiment"] == e["id"]) & (scores["label"] == l)].sort_values("score", ascending=False)
        for rank, (_, r) in enumerate(sub.iterrows(), 1):
            row = {"experiment": e["id"], "query | corpus": e["title"], "model": l, "rank": rank,
                   "window": int(r["window"]), "verses": r["verses"], "first word": int(r["start"]) + 1,
                   "last word": int(r["end"]), "score": round(float(r["score"]), 4), "most likely answer": r["top"],
                   "numeric-answer probability": round(float(r["digit_mass"]), 4)}
            row.update({f"p_{v}": round(float(p), 4) for v, p in enumerate(r["probs"])})
            row["text"] = e["windows"][int(r["window"]) - 1]["text"]
            rank_rows.append(row)
RANKINGS = pd.DataFrame(rank_rows)
if len(RANKINGS):
    RANKINGS.to_csv(OUT / "window_rankings.csv", index=False, encoding="utf-8-sig")

for e in EXPERIMENTS:
    print(f"\n{e['id']}: {e['title']} (top 10 windows per model; all of them are in window_rankings.csv)")
    if len(RANKINGS):
        top = RANKINGS[(RANKINGS["experiment"] == e["id"]) & (RANKINGS["rank"] <= 10)]
        display(top[["model", "rank", "window", "verses", "score", "numeric-answer probability", "text"]]
                .assign(text=lambda d: d["text"].str.slice(0, 80) + "…").set_index(["model", "rank"]))

### Summary

In [ ]:
def word_scores(e, label):
    """Each corpus word's score is the mean of the windows that contain it."""
    n = len(e["words"])
    total, count = np.zeros(n), np.zeros(n)
    for _, r in scores[(scores["experiment"] == e["id"]) & (scores["label"] == label)].iterrows():
        total[int(r["start"]):int(r["end"])] += r["score"]
        count[int(r["start"]):int(r["end"])] += 1
    with np.errstate(invalid="ignore"):
        return total / count

summary_rows = []
for e in EXPERIMENTS:
    for l in LABELS:
        sub = scores[(scores["experiment"] == e["id"]) & (scores["label"] == l)]
        if sub.empty:
            continue
        best = sub.loc[sub["score"].idxmax()]
        row = {"experiment": e["id"], "query | corpus": e["title"], "model": l, "windows": len(sub),
               "highest score": round(best["score"], 2), "highest window": f"{int(best['window'])} ({best['verses']})",
               "mean score": round(sub["score"].mean(), 2)}
        if l != LEX:
            row["numeric-answer probability"] = round(float(sub["digit_mass"].mean()), 3)
            row["seconds per window"] = round(float(sub["seconds"].mean()), 2)
        summary_rows.append(row)
SUMMARY = pd.DataFrame(summary_rows)
display(SUMMARY.set_index(["experiment", "model"]))
low = (SUMMARY[SUMMARY["numeric-answer probability"].fillna(1) < MIN_DIGIT_MASS]
       if "numeric-answer probability" in SUMMARY else SUMMARY.iloc[0:0])
if len(low):
    print(f"Low numeric-answer probability (< {MIN_DIGIT_MASS}) for {', '.join(sorted(set(low['model'])))}: "
          "the model mostly wanted to reply with words before a number, so treat its scores with caution.")
if MODEL_INFO:
    print("\nWhat each model wrote for each experiment's first window (it should be a bare number):")
    for m, i in MODEL_INFO.items():
        if m in MODELS:
            print(f"  {model_label(m)}: " + "; ".join(f"{k} “{v}”" for k, v in i.get("first_replies", {}).items()))

### Relatedness along the corpus

Each line gives every word the average score of the windows that contain it.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle
from matplotlib.textpath import TextToPath
from matplotlib.font_manager import FontProperties

INK, INK2, MUTED, GRID, AXIS, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
plt.rcParams.update({"font.family": "DejaVu Sans", "axes.edgecolor": AXIS, "axes.labelcolor": INK2,
                     "xtick.color": MUTED, "ytick.color": MUTED, "text.color": INK,
                     "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE})

def verse_ticks(e, max_ticks=40):
    starts = [0] + [i for i in range(1, len(e["words"])) if e["verse_idx"][i] != e["verse_idx"][i - 1]]
    min_gap = len(e["words"]) / (max_ticks * 1.15)
    pos, labels, last = [], [], -1e9
    for s in starts:
        if s - last >= min_gap:
            pos.append(s)
            labels.append(e["short_refs"][e["verse_idx"][s]])
            last = s
    return pos, labels

def plot_line(e, ax):
    n = len(e["words"])
    for l in LABELS:
        ys = word_scores(e, l)
        if np.all(np.isnan(ys)):
            continue
        lex = l == LEX
        ax.plot(np.arange(n) + 0.5, ys, color=COLOR[l], lw=1.2 if lex else 1.8, ls=(0, (4, 3)) if lex else "-",
                solid_capstyle="round", zorder=2 if lex else 3)
    ax.set_xlim(0, n)
    ax.set_ylim(0, 10.4)
    ax.set_yticks(range(0, 11, 2))
    pos, labels = verse_ticks(e)
    ax.set_xticks(pos)
    ax.set_xticklabels(labels, fontsize=7.5, ha="left")
    ax.grid(axis="y", color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.set_xlabel(e["corpus"]["name"] + (f" · {e['prefix'].rstrip(' :.')}" if e["prefix"] else ""), fontsize=9)
    ax.set_ylabel("relatedness (0–10)", fontsize=9)
    ax.set_title(f"{e['title']}", fontsize=11, loc="left")

def legend_handles():
    return [Line2D([], [], color=COLOR[l], lw=1.2 if l == LEX else 1.8, ls=(0, (4, 3)) if l == LEX else "-", label=l)
            for l in LABELS]

for e in EXPERIMENTS:
    fig, ax = plt.subplots(figsize=(14, 4.2))
    plot_line(e, ax)
    ax.legend(handles=legend_handles(), loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False, fontsize=8.5,
              labelcolor=INK2)
    fig.tight_layout()
    fig.savefig(OUT / f"{e['id']}_line.png", dpi=150)
    plt.show()

### The corpus with a bar above every word

Bar height is the word's score, from 0 (empty) to 10 (the full grey track), one bar per model. The small grey numbers are the references; the model never saw them.

In [ ]:
_T2P, _widths = TextToPath(), {}

def font(size):
    return FontProperties(family="DejaVu Sans", size=size)

def text_width(s, size):
    if (s, size) not in _widths:
        _widths[(s, size)] = _T2P.get_text_width_height_descent(s, font(size), ismath=False)[0] if s.strip() else 0.0
    return _widths[(s, size)]

def plot_text_with_bars(e, labels=None, fontsize=11, width_in=15, rows_per_page=110):
    with plt.rc_context({"text.hinting": "no_hinting", "text.hinting_factor": 1}):
        labels = [l for l in (labels or MODEL_LABELS) if not np.all(np.isnan(word_scores(e, l)))]
        per_word = {l: word_scores(e, l) for l in labels}
        words, vi = e["words"], e["verse_idx"]
        width_pt, left, right = width_in * 72, 30.0, 18.0
        usable, space, msize = width_pt - left - right, fontsize * 0.45, fontsize * 0.62
        rows, cur, pos = [], [], 0.0
        for i, w in enumerate(words):
            marker = e["short_refs"][vi[i]] if i == 0 or vi[i] != vi[i - 1] else ""
            mw = text_width(marker, msize) + 2 if marker else 0.0
            ww = max(text_width(w, fontsize), 4.0 * max(1, len(labels)))
            if cur and pos + mw + ww > usable:
                rows.append(cur)
                cur, pos = [], 0.0
            cur.append({"i": i, "marker": marker, "mx": left + pos, "x": left + pos + mw, "w": ww})
            pos += mw + ww + space
        rows.append(cur)
        band, text_h, gap, top = 30.0, fontsize * 1.45, 12.0, 52.0
        row_h = band + text_h + gap
        pages = [rows[k:k + rows_per_page] for k in range(0, len(rows), rows_per_page)]
        for p, page in enumerate(pages, 1):
            height_pt = top + len(page) * row_h + 8
            fig = plt.figure(figsize=(width_pt / 72, height_pt / 72))
            ax = fig.add_axes([0, 0, 1, 1])
            ax.set_xlim(0, width_pt)
            ax.set_ylim(height_pt, 0)
            ax.axis("off")
            for r, row in enumerate(page):
                y0 = top + r * row_h
                y1 = y0 + band
                base = y1 + fontsize * 1.05
                ax.text(left - 4, y0, "10", fontsize=6, color=MUTED, ha="right", va="center")
                ax.text(left - 4, y1, "0", fontsize=6, color=MUTED, ha="right", va="center")
                for c in row:
                    ax.add_patch(Rectangle((c["x"], y0), c["w"], band, color=GRID, alpha=0.45, lw=0, zorder=1))
                    nb = max(1, len(labels))
                    gb = 1.0 if nb > 1 else 0.0
                    bw = (c["w"] - gb * (nb - 1)) / nb
                    for k, l in enumerate(labels):
                        s = per_word[l][c["i"]]
                        if not np.isnan(s):
                            h = band * float(np.clip(s, 0, 10)) / 10
                            ax.add_patch(Rectangle((c["x"] + k * (bw + gb), y1 - h), bw, h, color=COLOR[l], lw=0, zorder=2))
                    ax.plot([c["x"], c["x"] + c["w"]], [y1, y1], color=AXIS, lw=0.6, zorder=3)
                    ax.text(c["x"], base, words[c["i"]], fontproperties=font(fontsize), color=INK, va="baseline", zorder=4)
                    if c["marker"]:
                        ax.text(c["mx"], base - fontsize * 0.55, c["marker"], fontproperties=font(msize), color=MUTED,
                                va="baseline", zorder=4)
            title = e["title"] + (f" (part {p} of {len(pages)})" if len(pages) > 1 else "")
            ax.text(left, 22, title, fontsize=12, color=INK, va="baseline")
            if labels:
                ax.legend(handles=[Patch(color=COLOR[l], lw=0, label=l) for l in labels], loc="upper right",
                          bbox_to_anchor=(1 - right / width_pt, 1 - 4 / height_pt), ncol=min(3, len(labels)),
                          frameon=False, fontsize=8.5, labelcolor=INK2)
            fig.savefig(OUT / (f"{e['id']}_text_bars" + (f"_part{p}" if len(pages) > 1 else "") + ".png"), dpi=130)
            plt.show()

for e in EXPERIMENTS:
    plot_text_with_bars(e)

### Download

In [ ]:
SUMMARY.to_csv(OUT / "summary.csv", index=False, encoding="utf-8-sig")
word_rows = []
for e in EXPERIMENTS:
    ws = {l: word_scores(e, l) for l in LABELS}
    for i, w in enumerate(e["words"]):
        word_rows.append({"experiment": e["id"], "word": i + 1, "reference": e["corpus"]["refs"][e["verse_idx"][i]],
                          "text": w, **{l: round(float(ws[l][i]), 4) for l in LABELS}})
pd.DataFrame(word_rows).to_csv(OUT / "word_scores.csv", index=False, encoding="utf-8-sig")
(OUT / "experiments.json").write_text(json.dumps(DATA, ensure_ascii=False, indent=1), encoding="utf-8")
with pd.ExcelWriter(OUT / "results.xlsx") as xl:
    SUMMARY.to_excel(xl, sheet_name="summary", index=False)
    for e in EXPERIMENTS:
        if len(RANKINGS):
            RANKINGS[RANKINGS["experiment"] == e["id"]].to_excel(xl, sheet_name=e["id"], index=False)

archive = shutil.make_archive(OUTPUT_DIR, "zip", OUTPUT_DIR)
print("Saved:", *sorted(p.name for p in OUT.iterdir()), sep="\n  ")
try:
    from google.colab import files
    files.download(archive)
except Exception:
    print("\nZipped to", archive)